# Cинтетические датасеты — бэйзлайн (дефолтные параметры)

Алгоритмы без подбора параметров.

In [1]:
import sys, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    adjusted_rand_score, adjusted_mutual_info_score,
    normalized_mutual_info_score, fowlkes_mallows_score,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
)

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
from data_generator.habr_synthetic import all_habr_datasets


In [2]:
from algorithms import (
    DBSCANWrapper, HDBSCANWrapper,
    DPCWrapper, RDDACWrapper, CKDPCWrapper,
)

ALG_NAMES = ['DBSCAN', 'HDBSCAN', 'DPC', 'RD-DAC', 'CKDPC']
ALG_CLASSES = {
    'DBSCAN':  DBSCANWrapper,
    'HDBSCAN': HDBSCANWrapper,
    'DPC':     DPCWrapper,
    'RD-DAC':  RDDACWrapper,
    'CKDPC':   CKDPCWrapper,
}


In [3]:
DATASETS = all_habr_datasets()

In [4]:
def compute_all_metrics(X, labels, y_true):
    labels  = np.asarray(labels, dtype=int)
    y_true  = np.asarray(y_true, dtype=int)
    mask_nn = labels != -1
    k_found   = len(set(labels[mask_nn].tolist())) if mask_nn.any() else 0
    noise_pct = float((~mask_nn).sum()) / len(labels) * 100
    if k_found >= 2:
        ari = float(adjusted_rand_score(y_true, labels))
        ami = float(adjusted_mutual_info_score(y_true, labels))
        nmi = float(normalized_mutual_info_score(y_true, labels))
        fmi = float(fowlkes_mallows_score(y_true, labels))
    elif k_found == 1:
        ari = ami = nmi = fmi = 0.0
    else:
        ari = ami = nmi = fmi = float('nan')
    X_sub, l_sub = X[mask_nn], labels[mask_nn]
    if mask_nn.sum() >= 2 and len(np.unique(l_sub)) >= 2:
        try:    sc  = float(silhouette_score(X_sub, l_sub))
        except: sc  = float('nan')
        try:    chi = float(calinski_harabasz_score(X_sub, l_sub))
        except: chi = float('nan')
        try:    dbi = float(davies_bouldin_score(X_sub, l_sub))
        except: dbi = float('nan')
    else:
        sc = chi = dbi = float('nan')
    return dict(k_found=k_found, noise_pct=noise_pct,
                ARI=ari, AMI=ami, NMI=nmi, FMI=fmi, SC=sc, CHI=chi, DBI=dbi)


def run_default(alg_class, X):
    try:
        return np.asarray(alg_class().fit_predict(X), dtype=int)
    except Exception as e:
        print(f'  ERROR: {e}')
        return np.zeros(len(X), dtype=int)


In [8]:
def show_results(all_results, datasets):
    summary = []
    for ds in datasets:
        dname = ds.name
        k_true = len(np.unique(ds.y_true))
        for alg in ALG_NAMES:
            m = all_results[dname][alg]
            def _v(k): return round(m[k], 4) if isinstance(m[k], float) and m[k]==m[k] else None
            summary.append({'Dataset': dname, 'k_true': k_true,
                            'Algorithm': alg, 'k_found': m['k_found'],
                            'noise%': round(m['noise_pct'],1),
                            'ARI': _v('ARI'), 'AMI': _v('AMI'), 'NMI': _v('NMI'),
                            'FMI': _v('FMI'), 'SC': _v('SC'), 'CHI': _v('CHI'), 'DBI': _v('DBI')})
    df = pd.DataFrame(summary)
    display(df.set_index(['Dataset','Algorithm']).style
            .format({'ARI':'{:.4f}','AMI':'{:.4f}','NMI':'{:.4f}',
                     'FMI':'{:.4f}','SC':'{:.4f}','DBI':'{:.4f}',
                     'noise%':'{:.1f}'}, na_rep='-')
            .background_gradient(subset=['ARI', 'AMI', 'NMI', 'FMI'], cmap='RdYlGn', vmin=0, vmax=1)
            .set_caption('Полная таблица метрик'))


In [9]:
ALL_RESULTS = {}
for ds in DATASETS:
    X, y = ds.X, ds.y_true
    print(f'\n{ds.name}  [{X.shape[0]} x {X.shape[1]}]  k_true={len(np.unique(y))}')
    rows = {}
    for alg_name in ALG_NAMES:
        t0     = time.time()
        labels = run_default(ALG_CLASSES[alg_name], X)
        mets   = compute_all_metrics(X, labels, y)
        elapsed = time.time() - t0
        rows[alg_name] = mets
        print(f'  {alg_name:<8}  k={mets["k_found"]}  noise={mets["noise_pct"]:.1f}%  '
              f'ARI={mets["ARI"]:.4f}  ({elapsed:.1f}s)')
    ALL_RESULTS[ds.name] = rows



habr_numpy_linear  [150 x 2]  k_true=3
  DBSCAN    k=12  noise=46.7%  ARI=0.0828  (0.0s)
  HDBSCAN   k=11  noise=4.7%  ARI=0.5259  (0.0s)
  DPC       k=5  noise=0.0%  ARI=0.8331  (0.0s)
  RD-DAC    k=2  noise=0.0%  ARI=0.0374  (0.0s)
  CKDPC     k=6  noise=8.0%  ARI=0.6463  (0.0s)

habr_numpy_timeseries  [150 x 2]  k_true=3
  DBSCAN    k=12  noise=6.7%  ARI=0.3304  (0.0s)
  HDBSCAN   k=9  noise=1.3%  ARI=0.6722  (0.0s)
  DPC       k=8  noise=0.0%  ARI=0.6108  (0.0s)
  RD-DAC    k=10  noise=0.0%  ARI=0.4183  (0.0s)
  CKDPC     k=4  noise=6.7%  ARI=0.8191  (0.0s)

habr_sklearn_blobs  [300 x 2]  k_true=3
  DBSCAN    k=9  noise=49.0%  ARI=0.1508  (0.0s)
  HDBSCAN   k=3  noise=0.0%  ARI=1.0000  (0.0s)
  DPC       k=5  noise=0.0%  ARI=0.7484  (0.0s)
  RD-DAC    k=6  noise=0.0%  ARI=0.7116  (0.0s)
  CKDPC     k=4  noise=7.7%  ARI=0.8754  (0.0s)

habr_sklearn_regression_style  [150 x 2]  k_true=3
  DBSCAN    k=10  noise=6.0%  ARI=0.4630  (0.0s)
  HDBSCAN   k=11  noise=2.0%  ARI=0.5354  (0.0s)

In [10]:
show_results(ALL_RESULTS, DATASETS)